# 01 - Confounding and propensity scores

When treatment is not randomised, the people who receive it differ from those
who do not — and those differences, not the treatment, may explain the outcome
gap. This notebook makes that failure concrete, then removes it two ways:
by reweighting the sample, and by matching comparable units.

Because the data are synthetic, the true effect is known, so every method can be
scored rather than merely compared with the others.

## Causal question

A programme is available to individuals, and uptake depends on their
characteristics. What is the average effect of participating on the outcome?

## Data and design

- **Unit of analysis:** one individual.
- **Treatment:** `treatment`, binary, with take-up driven by the covariates.
- **Outcome:** `outcome`, continuous.
- **Covariates:** `x1`, `x2`, `x3`, all measured before treatment.

`true_propensity` and `true_ite` are also present. They are used for checking,
never for estimating — an estimator that saw them would be answering a different
question from the one a real analysis faces.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SRC_PATH = PROJECT_ROOT / "src"
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression

from causal_inference_lab.data_generators import make_confounded_binary_treatment
from causal_inference_lab.diagnostics import balance_table, ipw_weights, overlap_summary
from causal_inference_lab.estimators import difference_in_means, g_computation_ate, ipw_ate
from causal_inference_lab.matching import propensity_score_matching
from causal_inference_lab.uncertainty import bootstrap_ate

COVARIATES = ["x1", "x2", "x3"]

dataset = make_confounded_binary_treatment(n=5_000, seed=42)
data = dataset.data

print(f"observations:  {len(data):,}")
print(f"treated share: {data['treatment'].mean():.1%}")
print(f"true ATE:      {dataset.true_ate:.3f}")
print()
print("covariate means by treatment arm:")
print(data.groupby("treatment")[COVARIATES].mean().to_string(float_format=lambda v: f"{v:.3f}"))

**Interpretation.** The arms are not comparable. Treated units have visibly
higher `x1` and lower `x2` than untreated ones. Any raw comparison of outcomes
will attribute the consequences of those differences to the treatment.

## Estimand

The **average treatment effect (ATE)**: the mean difference in outcomes if
everyone were treated versus if nobody were.

Note this in advance, because one estimator below silently targets something
else.

## Identification assumptions

1. **Conditional ignorability.** Given `x1`, `x2`, `x3`, treatment is as good as
   randomly assigned. Untestable; everything here depends on it.
2. **Overlap.** Every covariate profile has a non-zero chance of either arm.
   Checkable, and checked below.
3. **Consistency and no interference.** One version of the treatment, and no
   unit's treatment affects another's outcome.

Balance diagnostics can show that adjustment worked on *measured* covariates.
No diagnostic speaks to unmeasured ones.

## Estimation

Start with the naive contrast, which ignores confounding entirely, then adjust
by modelling the outcome and by modelling treatment assignment.

In [ ]:
naive = difference_in_means(data)
outcome_model = g_computation_ate(data, covariates=COVARIATES)
weighted = ipw_ate(data, covariates=COVARIATES)

results = pd.DataFrame(
    [
        ("naive difference in means", naive.estimate, naive.estimand),
        ("g-computation", outcome_model.estimate, outcome_model.estimand),
        ("IPW", weighted.estimate, weighted.estimand),
    ],
    columns=["method", "estimate", "estimand"],
)
results["error"] = results["estimate"] - dataset.true_ate

print(f"true ATE: {dataset.true_ate:.3f}\n")
print(results.to_string(index=False, float_format=lambda v: f"{v:.3f}"))

**Interpretation.** The naive estimate is 3.80 against a truth of 1.99 — it
overstates the effect by 91%. Both adjusted estimators land within 0.11 of the
truth.

They get there by different routes. G-computation models the outcome and
predicts what each unit would have done under either arm. IPW leaves the outcome
alone and reweights units so that the treated and untreated groups resemble the
full population. That they agree closely is mild evidence that neither model is
badly misspecified — but only mild, since both rest on the same covariates.

## Diagnostics

Two things to check before believing any of this: whether overlap holds, and
whether the reweighting actually balanced the covariates it was supposed to.

In [ ]:
propensity = (
    LogisticRegression(max_iter=1_000)
    .fit(data[COVARIATES], data["treatment"])
    .predict_proba(data[COVARIATES])[:, 1]
)

print("estimated propensity distribution:")
for name, value in overlap_summary(propensity).items():
    print(f"  {name:>7}: {value:.3f}")

weights = ipw_weights(data, covariates=COVARIATES)
print(f"\nlargest IPW weight: {weights.max():.1f}  (n = {len(weights):,})")
print(f"weight share held by the heaviest 1% of units: "
      f"{np.sort(weights)[-len(weights) // 100:].sum() / weights.sum():.1%}")

**Interpretation.** Overlap holds in the sense that matters — the bulk of the
distribution sits well away from 0 and 1, with the 5th and 95th percentiles at
0.13 and 0.87. The extremes are less comfortable: a minimum of 0.006 means some
units are almost never treated, and those units receive very large weights.

The heaviest 1% of units carry about 5% of the total weight, and the single
largest weight is 17.7. That is tolerable here, but it is the quantity to watch:
IPW estimates are effectively computed on a smaller sample than the row count
suggests, and one extreme weight can dominate the answer.

Balance is the direct test of whether weighting did its job. Standardized mean
differences below 0.1 in absolute value are the conventional threshold.

In [ ]:
before = balance_table(data, covariates=COVARIATES)
after = balance_table(data, covariates=COVARIATES, weights=weights)

comparison = before[["covariate", "smd"]].merge(
    after[["covariate", "smd"]], on="covariate", suffixes=("_before", "_after")
)
print(comparison.to_string(index=False, float_format=lambda v: f"{v:.3f}"))
print(f"\nworst |SMD| before weighting: {before['abs_smd'].max():.3f}")
print(f"worst |SMD| after weighting:  {after['abs_smd'].max():.3f}")

**Interpretation.** Before weighting, `x1` differs between arms by 0.78 standard
deviations — far beyond the 0.1 threshold, and `x2` at −0.57 is little better.
After weighting the worst imbalance is 0.031. The reweighted pseudo-population
is balanced on everything we measured.

This is the strongest evidence the notebook can offer, and it is worth being
precise about its limits: it demonstrates that the propensity model succeeded at
balancing `x1`, `x2`, `x3`. It is silent on any covariate not in that list, and
conditional ignorability is a claim about *all* confounders.

Matching offers a different route to the same goal, and produces an
interpretable matched sample rather than a set of weights.

In [ ]:
matched = propensity_score_matching(data, covariates=COVARIATES)

print(f"matching estimate: {matched.effect.estimate:.3f}")
print(f"estimand:          {matched.effect.estimand}")
print(f"units dropped:     {matched.dropped_units}")
print(f"\nIPW estimate:      {weighted.estimate:.3f}  (estimand {weighted.estimand})")
print(f"true ATE:          {dataset.true_ate:.3f}")

**Interpretation.** Matching returns 2.21 where IPW returned 2.10, and the
difference is not noise — the two are estimating **different quantities**. The
`estimand` field says so: matching reports the ATT, the effect among the treated,
while IPW reports the ATE.

Those coincide only when the effect is constant across units. Here it is not, so
the gap is real and interpretable: units who actually take up the programme
benefit somewhat more than the population would on average. Reading the matching
output as an ATE would be an error the result object is explicitly trying to
prevent.

## Uncertainty

Every number above is a point estimate. Bootstrapping the whole IPW procedure —
refitting the propensity model on each resample — gives an interval that
reflects both sampling variability and the instability of the weights.

In [ ]:
interval = bootstrap_ate(
    data,
    estimator=ipw_ate,
    covariates=COVARIATES,
    n_bootstrap_samples=300,
    seed=42,
)

print(f"IPW estimate:  {interval.estimate:.3f}")
print(f"95% interval:  [{interval.lower:.3f}, {interval.upper:.3f}]")
print(f"standard error:{interval.std_error:.3f}")
print(f"true ATE:      {dataset.true_ate:.3f}")
print(f"covers truth:  {interval.lower <= dataset.true_ate <= interval.upper}")
print(f"\nnaive estimate {naive.estimate:.3f} lies far outside this interval.")

**Interpretation.** The interval is [2.006, 2.180], and the true effect of 1.990
lies **outside it** — just below the lower bound. A nominally 95% interval has
missed the answer.

This is not a bug, and it is worth understanding rather than explaining away.
The bootstrap resamples the data and repeats the whole IPW procedure, so it
measures how much the estimate would move across samples *drawn from this
population, using this model*. It is centred on 2.097, not on the truth. IPW
carries a small upward bias here — from the propensity model's imperfection and
from those large weights — and a confidence interval quantifies variance around
whatever the estimator converges to. It cannot detect that the target is
displaced.

Two practical consequences. Narrow intervals are not the same as accurate ones:
this interval is tight (width 0.17) and wrong. And coverage statements are
conditional on the model being right — so an interval like this belongs in a
report alongside the assumptions it rests on, which is what notebook 06 builds.

The adjustment is still doing real work: the naive estimate of 3.80 is nowhere
near this interval, and the residual bias of 0.11 is a twentieth of the 1.81
that adjustment removed.

## Limitations

- **Conditional ignorability is assumed.** Balance on `x1`, `x2`, `x3` is
  evidence that adjustment worked, not evidence that these are all the
  confounders. Notebook 06 quantifies exposure to that failure.
- **Overlap is imperfect.** The minimum estimated propensity is 0.006, the
  largest weight is 17.7, and the heaviest 1% of units carry 5% of the weight.
  Estimates in that region rest on very few observations.
- **The bootstrap interval excluded the true effect.** IPW's small upward bias
  is not something a variance-based interval can see. Treat interval width as a
  statement about precision, never about accuracy.
- **Matching targets the ATT.** Comparing it to the ATE estimators is
  informative only because the difference is understood; it is not a
  disagreement about the same quantity.
- **Both models are linear.** They work here because the data are close to
  linear. Notebook 07 shows what happens when they are not, and the answer is
  worse than doing nothing.
- **Synthetic data.** No measurement error, no missingness, and covariates that
  are genuinely pre-treatment — three conveniences real data rarely provides.